<a href="https://colab.research.google.com/github/jonathancdelc/jonathancdelc/blob/main/Practica_4_Data_Science_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import sqlite3

def ejecutar_pipeline_con_tcl():
    # =========================================================================
    # PASO 1: Crear la base de datos y la tabla para el pipeline
    # =========================================================================
    # Conectamos a una base de datos en memoria para mantener el ejercicio limpio
    conexion = sqlite3.connect(":memory:")
    cursor = conexion.cursor()

    # Creamos una tabla simulando un pipeline de procesamiento de usuarios para ML
    cursor.execute("""
        CREATE TABLE pipeline_usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            etapa_pipeline TEXT,
            procesado_ok INTEGER
        )
    """)
    conexion.commit() # Confirmamos la estructura inicial
    print("Paso 1: Base de datos y tabla 'pipeline_usuarios' creadas con éxito.")

    try:
        # =========================================================================
        # PASO 2: Iniciar la transacción e insertar los primeros datos
        # =========================================================================
        # En sqlite3, ejecutar un comando de modificación inicia una transacción automáticamente.
        # Insertamos el primer lote de datos (Etapa 1: Carga inicial)
        print("\nPaso 2: Iniciando transacciones de carga inicial...")
        cursor.execute("INSERT INTO pipeline_usuarios (nombre, etapa_pipeline, procesado_ok) VALUES ('Ana', 'Carga', 1)")
        cursor.execute("INSERT INTO pipeline_usuarios (nombre, etapa_pipeline, procesado_ok) VALUES ('Carlos', 'Carga', 1)")

        # =========================================================================
        # PASO 3: Usar SAVEPOINT para crear un punto intermedio
        # =========================================================================
        # Creamos un punto de restauración llamado 'carga_inicial_ok'
        cursor.execute("SAVEPOINT carga_inicial_ok")
        print("Paso 3: SAVEPOINT 'carga_inicial_ok' creado exitosamente.")

        # =========================================================================
        # PASO 4: Simular la siguiente etapa y forzar un error (ROLLBACK parcial)
        # =========================================================================
        print("\nIniciando Etapa 2: Transformación de datos...")
        # Intentamos actualizar a Ana (esto saldrá bien)
        cursor.execute("UPDATE pipeline_usuarios SET etapa_pipeline = 'Transformacion' WHERE nombre = 'Ana'")

        # Simulamos un error del sistema de forma manual (ej. falla de red o formato inesperado)
        error_en_pipeline = True

        if error_en_pipeline:
            print("¡ALERTA! Se detectó un fallo crítico procesando los datos de Carlos.")
            # Aplicamos ROLLBACK parcial: regresamos exactamente al estado del SAVEPOINT
            cursor.execute("ROLLBACK TO SAVEPOINT carga_inicial_ok")
            print("Paso 4: ROLLBACK parcial aplicado. Volvimos al estado de 'carga_inicial_ok'.")

        # =========================================================================
        # PASO 5: Confirmar los cambios válidos con COMMIT
        # =========================================================================
        # Al hacer COMMIT, guardamos de forma permanente lo que sobrevivió al rollback
        conexion.commit()
        print("\nPaso 5: COMMIT ejecutado. Cambios guardados permanentemente.")

    except Exception as e:
        # Si ocurre un error inesperado fuera de nuestra lógica controlada, revertimos TODO
        print(f"Error inesperado fuera del flujo controlado: {e}")
        conexion.rollback()

    finally:
        # =========================================================================
        # Verificación Final del Estado de los Datos
        # =========================================================================
        print("\n--- ESTADO FINAL DE LA BASE DE DATOS ---")
        cursor.execute("SELECT * FROM pipeline_usuarios")
        resultados = cursor.fetchall()

        for fila in resultados:
            print(f"ID: {fila[0]} | Nombre: {fila[1]} | Etapa: {fila[2]} | OK: {fila[3]}")

        # Cerramos la conexión de forma segura
        conexion.close()

if __name__ == "__main__":
    ejecutar_pipeline_con_tcl()

Paso 1: Base de datos y tabla 'pipeline_usuarios' creadas con éxito.

Paso 2: Iniciando transacciones de carga inicial...
Paso 3: SAVEPOINT 'carga_inicial_ok' creado exitosamente.

Iniciando Etapa 2: Transformación de datos...
¡ALERTA! Se detectó un fallo crítico procesando los datos de Carlos.
Paso 4: ROLLBACK parcial aplicado. Volvimos al estado de 'carga_inicial_ok'.

Paso 5: COMMIT ejecutado. Cambios guardados permanentemente.

--- ESTADO FINAL DE LA BASE DE DATOS ---
ID: 1 | Nombre: Ana | Etapa: Carga | OK: 1
ID: 2 | Nombre: Carlos | Etapa: Carga | OK: 1


# Reflexión sobre el Impacto de TCL en la Consistencia
Si observas la salida del código en la consola, notarás algo muy interesante en el estado final:
*   Ana y Carlos aparecen en la base de datos (se mantuvieron los INSERT del Paso 2).
*   Ana vuelve a aparecer en la etapa 'Carga' en lugar de 'Transformacion'.





# ¿Por qué pasó esto?
Cuando ocurrió la simulación del error, el comando ROLLBACK TO SAVEPOINT carga_inicial_ok deshizo únicamente las acciones que ocurrieron después de haber creado ese punto de control. Por lo tanto, la actualización del UPDATE (que ponía a Ana en la etapa de Transformación) se borró por completo para garantizar que los datos no queden corruptos o inconsistentes tras el fallo de Carlos.
Gracias al COMMIT final, nos aseguramos de que el pipeline guarde el último estado 100% seguro y limpio, listo para que un modelo de Machine Learning lo consuma sin peligro de procesar datos corruptos.
¿Qué te parece este flujo? Si deseas, podemos modificar el ejercicio para ver cómo reaccionaría la base de datos si utilizáramos un ROLLBACK total en lugar de uno parcial.